# ckpt-chaos in Colab

Crash-tests a training script's checkpoints: it kills the script at every save point, resumes it, and reports whether the resumed run matches the fault-free run.

Runs on your own Colab runtime. Nothing is uploaded anywhere.

Step 1 installs the tool. Step 2 gets a script. Step 3 runs the test.

In [ ]:
!pip install -q "git+https://github.com/rkb32/ckpt-chaos.git"

## Step 2: get a script

Option A: use the example trainer from the repo (saves like most scripts do, so it is expected to fail).

Option B: upload your own script. Your script must take an output directory argument, save checkpoints into it, and resume from them if they exist. Set `SCRIPT` to its path and `SCRIPT_ARGS` to the arguments it takes, with `{out}` where the output directory goes.

In [ ]:
import subprocess

# Option A: the example trainer from the repo
subprocess.run(["git", "clone", "-q", "https://github.com/rkb32/ckpt-chaos.git", "/content/ckpt-chaos-src"], check=True)
SCRIPT = "/content/ckpt-chaos-src/examples/naive_torch.py"
SCRIPT_ARGS = ["--out", "{out}"]

# Option B: uncomment to upload your own script instead
# from google.colab import files
# up = files.upload()
# SCRIPT = "/content/" + next(iter(up))
# SCRIPT_ARGS = ["--out", "{out}"]  # edit to match your script

## Step 3: run the test

The example trainer prints `resumed from step N` on resume. For your own script, change `STEP_REGEX` to match whatever it prints, or set it to `None` to skip the step check.

The verdict table prints when the run finishes. `exit 0` means every crash point passed.

In [ ]:
import subprocess

STEP_REGEX = r"resumed from step (\d+)"
RESULT_FILE = "{out}/final.pt"  # set to None if your script does not write one

cmd = ["ckpt-chaos", "run", "--work", "/content/ckpt-chaos-runs"]
if STEP_REGEX:
    cmd += ["--step-regex", STEP_REGEX]
if RESULT_FILE:
    cmd += ["--result-file", RESULT_FILE]
cmd += ["--", "python", SCRIPT] + SCRIPT_ARGS

result = subprocess.run(cmd)
print("exit", result.returncode)